In [ ]:
import os
import time
import csv
import pickle
import tempfile
import requests
import json
import pandas as pd
import google.generativeai as genai
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==============================
# CONFIG
# ==============================
COOKIE_FILE = "naukri_cookies.pkl"
LOGIN_LINK_ID = "login_Layer"
NAUKRI_HOMEPAGE = "https://www.naukri.com/"
PROFILE_URL = "https://www.naukri.com/mnjuser/profile?id=&altresid"
RESUME_DRIVE_URL = "https://drive.google.com/file/d/1QgFWJDJS84TmvyRJeapjRUEtcEn_6QL9/view?usp=sharing"
EXTERNAL_CSV_PATH = "csv/naukri_external_apply.csv"
RESUME_JSON_PATH = "resumes/Yeswanth_Yerra_CV_structured.json"
JOBS_CSV_PATH = "csv/naukri_jobs.csv"

# ==============================
# GEMINI SETUP
# ==============================
api_key = os.getenv("GEMINI_API_KEY")
if api_key:
    genai.configure(api_key=api_key)
    print("Gemini API key configured successfully ✅")
else:
    print("⚠️ GEMINI_API_KEY not found.")

def ask_gemini(prompt, model="gemini-2.5-flash", temperature=0):
    model = genai.GenerativeModel(model_name=model)
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )
    return response.text

# ==============================
# DRIVER & COOKIE HELPERS
# ==============================
def setup_driver(headless=False):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
    options.add_argument("--start-maximized")
    options.add_argument("--no-sandbox")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def save_cookies(driver, cookie_file=COOKIE_FILE):
    with open(cookie_file, "wb") as f:
        pickle.dump(driver.get_cookies(), f)
    print(f"✅ Cookies saved to {cookie_file}")

def load_cookies(driver, cookie_file=COOKIE_FILE):
    with open(cookie_file, "rb") as f:
        cookies = pickle.load(f)
    driver.get(NAUKRI_HOMEPAGE)
    for c in cookies:
        try:
            driver.add_cookie(c)
        except Exception:
            pass
    driver.refresh()
    time.sleep(2)
    print("🍪 Cookies loaded and refreshed page.")

# ==============================
# LOGIN HELPERS
# ==============================
def is_logged_in(driver, timeout=8):
    try:
        WebDriverWait(driver, timeout).until(EC.invisibility_of_element_located((By.ID, LOGIN_LINK_ID)))
        return True
    except Exception:
        return False

def login_with_credentials(driver, wait, email, password):
    print("🔐 Logging in with credentials...")
    try:
        login_link = wait.until(EC.element_to_be_clickable((By.ID, LOGIN_LINK_ID)))
        login_link.click()
    except Exception:
        driver.get("https://login.naukri.com/nLogin/Login.php")

    email_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[contains(@placeholder,'Email')]")))
    pwd_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[@type='password']")))
    email_input.send_keys(email)
    pwd_input.send_keys(password)
    pwd_input.send_keys(Keys.RETURN)

    print("⏳ Waiting for login to complete...")
    if is_logged_in(driver):
        print("✅ Logged in successfully!")
    else:
        raise RuntimeError("❌ Login failed — check credentials or captcha.")

# ==============================
# RESUME UPLOAD
# ==============================
def download_resume_from_drive(drive_url):
    file_id = drive_url.split("/d/")[1].split("/")[0]
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    resp = requests.get(download_url, stream=True)
    if resp.status_code != 200:
        raise Exception("Resume download failed.")
    temp_path = os.path.join(tempfile.gettempdir(), "resume-drive.pdf")
    with open(temp_path, "wb") as f:
        for chunk in resp.iter_content(8192):
            f.write(chunk)
    print(f"📄 Resume downloaded to {temp_path}")
    return temp_path

def upload_resume_on_naukri(driver, wait):
    print("🚀 Uploading resume...")
    driver.get(PROFILE_URL)
    time.sleep(2)
    upload_input = wait.until(EC.presence_of_element_located((By.ID, "attachCV")))
    resume_path = download_resume_from_drive(RESUME_DRIVE_URL)
    upload_input.send_keys(resume_path)
    print("📤 Uploaded resume file input.")
    time.sleep(2)
    if os.path.exists(resume_path):
        os.remove(resume_path)
    print("✅ Resume upload flow done.")

# ==============================
# CHATBOT HANDLER
# ==============================
def handle_chatbot_questions(driver, wait, resume_json):
    print("🤖 Chatbot detected, answering questions...")
    while True:
        try:
            question_elems = driver.find_elements(By.CSS_SELECTOR, ".botMsg span")
            if not question_elems:
                break
            question = question_elems[-1].text.strip()
            if not question:
                time.sleep(1)
                continue
            print(f"💬 Question: {question}")

            option_labels = driver.find_elements(By.CSS_SELECTOR, ".ssrc__radio-btn-container label")
            options = [o.text.strip() for o in option_labels] if option_labels else []

            prompt = f"""
You are an AI answering recruiter questions for a Naukri job application.
Candidate resume JSON:
{json.dumps(resume_json)}

Recruiter question: {question}
Options: {options}

Rules:
- Choose an option if it matches logically.
- If not, answer concisely (max 2 lines).
- If the question asks about experience → answer only in years (numeric).
- If the question asks about salary or CTC → answer only in LPA (numeric).
- Pick one of the given options if available.
- Otherwise provide a short, relevant answer (max 2 lines).
- Be professional and accurate.
- incase of salary, provide the answer in lakhs (e.g., 4 or 5).
- incase of experience, round off years (2y 8m -> 3).
- For text/textarea: concise and truthful.
- For select/radio: best align with candidate’s skills.
"""
            try:
                answer = ask_gemini(prompt).strip()
            except Exception:
                answer = "Skip"

            print(f"🧠 Answer: {answer}")

            if option_labels:
                matched = False
                for o in option_labels:
                    if answer.lower() in o.text.lower():
                        driver.execute_script("arguments[0].click();", o)
                        matched = True
                        break
                if not matched:
                    for o in option_labels:
                        if "skip" in o.text.lower():
                            driver.execute_script("arguments[0].click();", o)
                            break
            else:
                text_box = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "[contenteditable='true']")))
                driver.execute_script("""
                    el = arguments[0];
                    el.innerText = arguments[1];
                    el.dispatchEvent(new Event('input', { bubbles: true }));
                """, text_box, answer)
                time.sleep(0.8)
                try:
                    save_btn = driver.find_element(By.CSS_SELECTOR, "div.sendMsg")
                    driver.execute_script("arguments[0].click();", save_btn)
                except:
                    pass
            time.sleep(2)
        except Exception:
            break
    print("✅ Chatbot Q&A completed.")

# ==============================
# JOB APPLY HANDLER
# ==============================
def save_external_job(job_url):
    os.makedirs(os.path.dirname(EXTERNAL_CSV_PATH), exist_ok=True)
    with open(EXTERNAL_CSV_PATH, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([job_url])
    print(f"💾 External job saved: {job_url}")

def apply_on_naukri(driver, wait, job_url, resume_json):
    print(f"💼 Visiting job: {job_url}")
    driver.get(job_url)
    time.sleep(4)

    try:
        # Support both 'Apply' and 'Apply on company site'
        apply_button = None
        try:
            apply_button = driver.find_element(By.ID, "apply-button")
        except:
            try:
                apply_button = driver.find_element(By.ID, "company-site-button")
            except:
                pass

        if not apply_button:
            print("⚠️ No apply button found, skipping.")
            return

        btn_text = apply_button.text.strip().lower()
        print(f"🧭 Button text: {btn_text}")

        if "company site" in btn_text:
            print("🔗 Detected external job, saving.")
            save_external_job(job_url)
            return

        if "apply" in btn_text:
            driver.execute_script("arguments[0].scrollIntoView(true);", apply_button)
            driver.execute_script("arguments[0].click();", apply_button)
            print("✅ Clicked Apply.")
            time.sleep(4)
            if driver.find_elements(By.CSS_SELECTOR, ".chatbot_MessageContainer"):
                handle_chatbot_questions(driver, wait, resume_json)
            else:
                print("ℹ️ No chatbot, single-click apply.")
        else:
            print("⚙️ Button unrecognized, skipping.")
    except Exception as e:
        print(f"❌ Error applying: {e}")

# ==============================
# MULTI-JOB HANDLER
# ==============================
def process_jobs_from_csv(csv_path, driver, wait, resume_json):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    if "links" not in df.columns:
        raise ValueError("CSV must have a 'links' column.")

    print(f"📋 {len(df)} jobs found in CSV.")
    for idx, row in df.iterrows():
        job_url = str(row["links"]).strip()
        if not job_url.startswith("http"):
            continue
        print(f"\n{'='*80}\n🔗 [{idx+1}/{len(df)}] {job_url}")
        try:
            upload_resume_on_naukri(driver, wait)
            apply_on_naukri(driver, wait, job_url, resume_json)
        except Exception as e:
            print(f"⚠️ Job error: {e}")
        time.sleep(2)
    print("\n🎯 All jobs processed successfully.")

# ==============================
# LOGIN + MAIN ENTRY
# ==============================
def ensure_logged_in(headless=False):
    email = os.getenv("NAUKRI_EMAIL")
    password = os.getenv("NAUKRI_PASSWORD")
    if not email or not password:
        raise RuntimeError("Set NAUKRI_EMAIL and NAUKRI_PASSWORD env vars.")

    driver = setup_driver(headless)
    wait = WebDriverWait(driver, 15)

    if os.path.exists(COOKIE_FILE):
        try:
            load_cookies(driver)
            if is_logged_in(driver):
                print("✅ Logged in via cookies.")
                return driver, wait
        except Exception as e:
            print(f"⚠️ Cookie login failed: {e}")

    driver.get(NAUKRI_HOMEPAGE)
    login_with_credentials(driver, wait, email, password)
    save_cookies(driver)
    return driver, wait

# ==============================
# MAIN EXECUTION
# ==============================
if __name__ == "__main__":
    try:
        with open(RESUME_JSON_PATH, "r", encoding="utf-8") as f:
            resume_json = json.load(f)
        print(f"📘 Loaded resume JSON from {RESUME_JSON_PATH}")

        driver, wait = ensure_logged_in(headless=False)
        process_jobs_from_csv(JOBS_CSV_PATH, driver, wait, resume_json)

        driver.quit()
        print("✅ Done with all job applications.")
    except Exception as e:
        print(f"❌ Fatal error: {e}")


Gemini API key configured successfully ✅
📘 Loaded resume JSON from resumes/Yeswanth_Yerra_CV_structured.json
🍪 Cookies loaded and refreshed page.
✅ Logged in via cookies.
📋 140 jobs found in CSV.

🔗 [1/140] https://www.naukri.com/job-listings-machine-learning-engineer-ii-swiggy-hyderabad-ahmedabad-bengaluru-3-to-5-years-231025501971
🚀 Uploading resume...
📄 Resume downloaded to /tmp/resume-drive.pdf
📤 Uploaded resume file input.
✅ Resume upload flow done.
💼 Visiting job: https://www.naukri.com/job-listings-machine-learning-engineer-ii-swiggy-hyderabad-ahmedabad-bengaluru-3-to-5-years-231025501971
⚠️ No apply button found, skipping.

🔗 [2/140] https://www.naukri.com/job-listings-machine-learning-engineer-phenom-people-hyderabad-5-to-8-years-110825008990
🚀 Uploading resume...
📄 Resume downloaded to /tmp/resume-drive.pdf
📤 Uploaded resume file input.
✅ Resume upload flow done.
💼 Visiting job: https://www.naukri.com/job-listings-machine-learning-engineer-phenom-people-hyderabad-5-to-8-years

E0000 00:00:1761840237.831577   14306 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


🧠 Answer: The candidate has no professional work experience specifically in AIML. However, they have strong project experience in Machine Learning and AI.
💬 Question: How many years of experience do you have in Artificial Intelligence?
🧠 Answer: The candidate has approximately 1 year and 2 months of project-based experience in Artificial Intelligence and Machine Learning, demonstrated through projects like the Movie Recommendation System, Network Intrusion Detection System, and Simple Song Search.
💬 Question: How many years of experience do you have in Machine Learning?
🧠 Answer: Yeswanth has approximately 1 year of project-based experience in Machine Learning, demonstrated through projects like the Movie Recommendation System and Network Intrusion Detection System.
💬 Question: Thank you for your responses.
🧠 Answer: You're welcome! I appreciate your time and look forward to hearing from you.
✅ Chatbot Q&A completed.

🔗 [7/140] https://www.naukri.com/job-listings-data-scientist-machine

KeyboardInterrupt: 